# Tutorial 4: DOE Planning (`grid` vs `sobol`)

Estimated time: 20-35 minutes

## Prerequisites
No optional dependencies required.

## Learning aims
- Primary package aim: compare DOE strategies and understand planning outputs
- Secondary scientific aim: reason about coverage, budget, and bias in simulation campaigns

## Success criteria
- you can justify a DOE strategy for a given model dimensionality and run budget


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Run both planning commands


In [ ]:
%%bash
set -euo pipefail
ROOT="$(pwd)"
if [ -d tutorials/specs ] && [ -d src ]; then ROOT="$(pwd)"
elif [ -d specs ] && [ -d ../src ]; then ROOT="$(pwd)/.."; fi
cd "$ROOT"
PYTHONPATH=src python -m bayesian_metamodeling.cli.main plan tutorials/specs/model.toy.grid.json
PYTHONPATH=src python -m bayesian_metamodeling.cli.main plan tutorials/specs/model.toy.sobol.json

## Step 2: Plot DOE points (graphic)


In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

grid_spec = json.loads((root / 'tutorials/specs/model.toy.grid.json').read_text())
sobol_spec = json.loads((root / 'tutorials/specs/model.toy.sobol.json').read_text())

# grid points
a_vals = grid_spec['design']['grid']['a']
b_vals = grid_spec['design']['grid']['b']
grid_pts = np.array([(a, b) for a in a_vals for b in b_vals], dtype=float)

# approximate sobol points from existing runs if available
runs_root = root / 'tmp/tutorials/toy_store_sobol/runs'
sobol_pts = []
if runs_root.exists():
    for run_dir in sorted(p for p in runs_root.iterdir() if p.is_dir()):
        inp = json.loads((run_dir / 'inputs.json').read_text())
        sobol_pts.append((inp['a'], inp['b']))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(grid_pts[:, 0], grid_pts[:, 1], s=90)
plt.title('Grid DOE points')
plt.xlabel('a')
plt.ylabel('b')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
if sobol_pts:
    sobol_arr = np.asarray(sobol_pts, dtype=float)
    plt.scatter(sobol_arr[:, 0], sobol_arr[:, 1], c='tab:orange', s=90)
    plt.title('Sobol DOE points (from runs)')
else:
    plt.text(0.1, 0.5, 'No Sobol runs yet\nRun `mm run tutorials/specs/model.toy.sobol.json`', fontsize=10)
    plt.title('Sobol DOE points')
plt.xlabel('a')
plt.ylabel('b')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Scientific checkpoint
Write a short decision note:
- when grid is better,
- when sobol is better,
- which one you would pick for a 6-10D study and why.
